# DocSensei: Enterprise Multi-Document Adaptive RAG Assistant

**DocSensei** is a production-grade Retrieval-Augmented Generation assistant that:

- Ingests multiple **PDF / DOCX / PPTX / TXT** files in a single session
- Answers questions **only** from the uploaded documents, with **page + file citations**
- Uses an **Adaptive RAG Decision Layer** to route queries (simple / complex / ambiguous) to the right retrieval strategy
- Combines **dense (Pinecone) + sparse (BM25) hybrid retrieval**
- Applies **contextual compression**, **query rewriting**, and **conversation memory**
- Serves everything through a modern **Gradio** chat interface

**Tech stack:** Python, LangChain, Pinecone, HuggingFace (`all-MiniLM-L6-v2`), Groq `llama-3.3-70b-versatile`, Gradio, `python-dotenv`.

> Run cells top to bottom in Google Colab. You will need a `.env` file (or Colab secrets) with `PINECONE_API_KEY` and `GROQ_API_KEY`.


# Install Dependencies

In [1]:
# %%capture
# !pip install -q groq pinecone-client rank_bm25 sentence-transformers
# !pip install -q pypdf python-docx python-pptx
# !pip install -q pytesseract pdf2image pillow
# !pip install -q python-dotenv
# !apt-get -qq install -y poppler-utils tesseract-ocr


# Imports

In [2]:
import io
import os
import re
import json
import time
import logging
import unicodedata
import uuid
from collections import deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any

from dotenv import load_dotenv

# Dependency-light implementations (no LangChain) shared with app.py
from pypdf import PdfReader
from docx import Document as DocxDocument
from docx.oxml.ns import qn
from pptx import Presentation
from pptx.shapes.group import GroupShape

from sentence_transformers import CrossEncoder, SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq
from pinecone import Pinecone, ServerlessSpec

# Optional OCR fallback for scanned PDFs
try:
    import pytesseract
    from pdf2image import convert_from_path
    OCR_LIBS_AVAILABLE = True
except ImportError:
    OCR_LIBS_AVAILABLE = False

#import gradio as gr

C:\Users\monis\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuration

In [3]:
# Embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

# Reranker
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANK_CANDIDATE_POOL = 15
RERANK_TOP_N = 8
COMPRESSION_TOP_N = 5

# Groq LLM
GROQ_MODEL = "llama-3.3-70b-versatile"
GROQ_TEMPERATURE = 0.1

# Pinecone
PINECONE_INDEX_NAME = "docsensei-index"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"

# Text chunking
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

# Hybrid retrieval
DENSE_WEIGHT = 0.6
BM25_WEIGHT = 0.4

# Conversation memory
MEMORY_WINDOW = 6

# OCR fallback
ENABLE_OCR_FALLBACK = True
OCR_DPI = 200
OCR_MIN_CHARS_PER_PAGE = 20

# Query categories used by the Adaptive RAG router
QUERY_LABELS = (
    "Simple",
    "Complex",
    "Comparison",
    "Summarization",
    "Reasoning",
    "Multi-document",
    "Follow-up",
)

# Number of documents retrieved for each query category
ROUTER_TOP_K = {
    "Simple": 3,
    "Complex": 8,
    "Comparison": 8,
    "Summarization": 10,
    "Reasoning": 8,
    "Multi-document": 10,
    "Follow-up": 5,
}

# API Setup (.env)

In [4]:
# Load environment variables from the .env file
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not PINECONE_API_KEY or not GROQ_API_KEY:
    raise ValueError(
        "Please add your PINECONE_API_KEY and GROQ_API_KEY to the .env file before running this notebook."
    )

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


# Utility Functions

In [5]:
def clean_text(text: str) -> str:
    """Clean and normalize extracted text while preserving its structure."""
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    lines = [line.strip() for line in text.split("\n")]
    cleaned_lines = []
    previous_blank = False

    for line in lines:
        if line == "":
            if previous_blank:
                continue
            previous_blank = True
        else:
            previous_blank = False
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()


def file_extension(filename: str) -> str:
    """Return the file extension in lowercase."""
    return os.path.splitext(filename)[1].lower()


def timeit(label: str):
    """Measure the execution time of a code block."""

    class _Timer:
        def __enter__(self):
            self.start = time.time()
            return self

        def __exit__(self, *exc):
            elapsed = time.time() - self.start
            print(f"{label} completed in {elapsed:.2f} seconds.")

    return _Timer()

# Document Loaders

In [6]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".pptx", ".txt"}


def ocr_is_available() -> bool:
    """Check whether the OCR dependencies are available for scanned PDF processing."""
    if not OCR_LIBS_AVAILABLE or not ENABLE_OCR_FALLBACK:
        return False

    try:
        pytesseract.get_tesseract_version()
        return True
    except Exception as error:
        print(f"Warning: OCR support is unavailable ({error}). Scanned PDF pages may not contain extractable text.")
        return False


OCR_AVAILABLE = ocr_is_available()


def _ocr_pdf_page(filepath: str, page_number: int) -> str:
    """Extract text from a single PDF page using OCR."""
    if not OCR_AVAILABLE:
        return ""

    try:
        images = convert_from_path(filepath, dpi=OCR_DPI, first_page=page_number, last_page=page_number)
        if not images:
            return ""
        return pytesseract.image_to_string(images[0]) or ""
    except Exception as error:
        print(f"Warning: OCR failed for {os.path.basename(filepath)} (Page {page_number}): {error}")
        return ""


def load_pdf(filepath: str) -> List[Dict[str, Any]]:
    """
    Extract text from each page of a PDF document. Pages with little or no
    extractable text are automatically processed using OCR when available.
    """
    pages: List[Dict[str, Any]] = []
    ocr_pages = 0

    try:
        reader = PdfReader(filepath)

        for page_number, page in enumerate(reader.pages, start=1):
            try:
                text = page.extract_text() or ""
            except Exception as error:
                print(f"Warning: Unable to extract text from page {page_number}: {error}")
                text = ""

            if len(text.strip()) < OCR_MIN_CHARS_PER_PAGE and OCR_AVAILABLE:
                ocr_text = _ocr_pdf_page(filepath, page_number)
                if len(ocr_text.strip()) > len(text.strip()):
                    text = ocr_text
                    ocr_pages += 1

            if text.strip():
                pages.append({"text": text, "page": page_number})

        print(f"Loaded '{os.path.basename(filepath)}' with {len(pages)} page(s). OCR applied to {ocr_pages} page(s).")

    except Exception as error:
        print(f"Unable to load PDF '{filepath}': {error}")

    return pages

In [7]:
def _iter_docx_headers_footers(document):
    """Yield the text from headers and footers across all document sections."""
    for section_index, section in enumerate(document.sections, start=1):
        for label, part in (
            (f"Header (Section {section_index})", section.header),
            (f"First-page Header (Section {section_index})", section.first_page_header),
            (f"Footer (Section {section_index})", section.footer),
            (f"First-page Footer (Section {section_index})", section.first_page_footer),
        ):
            try:
                text = "\n".join(paragraph.text for paragraph in part.paragraphs if paragraph.text.strip())
            except Exception:
                text = ""

            if text.strip():
                yield label, text


def _extract_docx_textboxes(document) -> List[str]:
    """Extract text from text boxes stored in the document XML."""
    textbox_texts: List[str] = []

    try:
        body = document.element.body

        for textbox in body.iter(qn("w:txbxContent")):
            paragraphs = textbox.iter(qn("w:p"))
            texts = []

            for paragraph in paragraphs:
                runs = paragraph.iter(qn("w:t"))
                run_text = "".join(text.text or "" for text in runs)
                if run_text.strip():
                    texts.append(run_text)

            textbox_content = "\n".join(texts)
            if textbox_content.strip():
                textbox_texts.append(textbox_content)

    except Exception as error:
        print(f"Warning: Unable to extract text boxes: {error}")

    return textbox_texts


def load_docx(filepath: str) -> List[Dict[str, Any]]:
    """Extract text from a DOCX document, including headings, tables, headers, footers, and text boxes."""
    pages: List[Dict[str, Any]] = []

    try:
        document = DocxDocument(filepath)
        content_parts: List[str] = []

        # Extract paragraphs
        for paragraph in document.paragraphs:
            text = paragraph.text.strip()
            if not text:
                continue

            style_name = (paragraph.style.name or "").lower() if paragraph.style else ""
            if "heading" in style_name or "title" in style_name:
                content_parts.append(f"## {text}")
            else:
                content_parts.append(text)

        # Extract tables
        for table_index, table in enumerate(document.tables, start=1):
            table_rows = []
            for row in table.rows:
                cells = [cell.text.strip() for cell in row.cells if cell.text.strip()]
                if cells:
                    table_rows.append(" | ".join(cells))

            if table_rows:
                content_parts.append(f"[Table {table_index}]\n" + "\n".join(table_rows))

        # Extract headers and footers
        for label, text in _iter_docx_headers_footers(document):
            content_parts.append(f"[{label}] {text}")

        # Extract text boxes
        for textbox_index, textbox_text in enumerate(_extract_docx_textboxes(document), start=1):
            content_parts.append(f"[Text Box {textbox_index}] {textbox_text}")

        full_text = "\n".join(part for part in content_parts if part.strip())

        if full_text.strip():
            pages.append({"text": full_text, "page": 1})

        print(f"Loaded '{os.path.basename(filepath)}' with {len(content_parts)} content block(s).")

    except Exception as error:
        print(f"Unable to load DOCX '{filepath}': {error}")

    return pages

In [8]:
def _iter_pptx_shapes(shapes):
    """Iterate through all shapes, including grouped shapes."""
    for shape in shapes:
        if isinstance(shape, GroupShape) or getattr(shape, "shape_type", None) == 6:
            try:
                yield from _iter_pptx_shapes(shape.shapes)
                continue
            except Exception:
                pass
        yield shape


def _extract_pptx_chart_labels(shape) -> List[str]:
    """Extract category and series labels from a chart."""
    labels: List[str] = []

    try:
        if not shape.has_chart:
            return labels

        chart = shape.chart

        for plot in chart.plots:
            try:
                categories = [str(category) for category in plot.categories if category is not None]
                labels.extend(categories)
            except Exception:
                pass

            for series in plot.series:
                series_name = getattr(series, "name", None)
                if series_name:
                    labels.append(str(series_name))

    except Exception as error:
        print(f"Warning: Unable to extract chart labels: {error}")

    return labels


def load_pptx(filepath: str) -> List[Dict[str, Any]]:
    """Extract text from each slide of a PowerPoint presentation."""
    pages: List[Dict[str, Any]] = []

    try:
        presentation = Presentation(filepath)

        for slide_number, slide in enumerate(presentation.slides, start=1):
            slide_content: List[str] = []

            for shape in _iter_pptx_shapes(slide.shapes):
                if getattr(shape, "has_text_frame", False):
                    is_title = False
                    try:
                        is_title = shape.placeholder_format is not None and shape.placeholder_format.idx == 0
                    except Exception:
                        is_title = False

                    for paragraph in shape.text_frame.paragraphs:
                        line = "".join(run.text for run in paragraph.runs).strip()
                        if not line and paragraph.text.strip():
                            line = paragraph.text.strip()
                        if line:
                            slide_content.append(f"# {line}" if is_title else line)

                chart_labels = _extract_pptx_chart_labels(shape)
                if chart_labels:
                    slide_content.append("[Chart] " + ", ".join(chart_labels))

            try:
                if slide.has_notes_slide:
                    notes = slide.notes_slide.notes_text_frame.text.strip()
                    if notes:
                        slide_content.append(f"[Speaker Notes] {notes}")
            except Exception as error:
                print(f"Warning: Unable to extract speaker notes from slide {slide_number}: {error}")

            slide_text = "\n".join(slide_content)
            if slide_text.strip():
                pages.append({"text": slide_text, "page": slide_number})

        print(f"Loaded '{os.path.basename(filepath)}' with {len(pages)} slide(s).")

    except Exception as error:
        print(f"Unable to load PPTX '{filepath}': {error}")

    return pages

In [9]:
def load_txt(filepath: str) -> List[Dict[str, Any]]:
    """Extract text from a plain text file."""
    pages: List[Dict[str, Any]] = []

    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as file:
            text = file.read()

        if text.strip():
            pages.append({"text": text, "page": 1})

        print(f"Loaded '{os.path.basename(filepath)}' successfully.")

    except Exception as error:
        print(f"Unable to load TXT file '{filepath}': {error}")

    return pages


LOADER_MAP = {
    ".pdf": load_pdf,
    ".docx": load_docx,
    ".pptx": load_pptx,
    ".txt": load_txt,
}


def load_document(filepath: str) -> Tuple[str, str, List[Dict[str, Any]]]:
    """Load a single document and return its filename, type, and extracted pages."""
    file_type = file_extension(filepath)
    filename = os.path.basename(filepath)

    if file_type not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported file type: {file_type}")

    print(f"Loading '{filename}'...")
    loader = LOADER_MAP[file_type]
    pages = loader(filepath)

    return filename, file_type.lstrip("."), pages


def load_documents(filepaths: List[str]) -> Dict[str, Tuple[str, List[Dict[str, Any]]]]:
    """Load multiple documents while continuing if an individual file fails."""
    loaded_documents: Dict[str, Tuple[str, List[Dict[str, Any]]]] = {}

    for file_path in filepaths:
        try:
            filename, file_type, pages = load_document(file_path)
            if not pages:
                print(f"Warning: No extractable text found in '{filename}'.")
                continue
            loaded_documents[filename] = (file_type, pages)
        except Exception as error:
            print(f"Skipping '{os.path.basename(file_path)}': {error}")

    print(f"Successfully loaded {len(loaded_documents)} out of {len(filepaths)} document(s).")

    return loaded_documents

# Metadata Extraction

In [10]:
def build_raw_documents(loaded: Dict[str, Tuple[str, List[Dict[str, Any]]]]) -> Dict[str, str]:
    """Combine the cleaned text from all pages of each document into a single string."""
    raw_documents: Dict[str, str] = {}

    for filename, (file_type, pages) in loaded.items():
        full_text = "\n\n".join(clean_text(page["text"]) for page in pages)
        if full_text.strip():
            raw_documents[filename] = full_text
        else:
            print(f"Warning: No extractable text found in '{filename}' after cleaning.")

    print(f"Prepared {len(raw_documents)} document(s) for chunking.")

    return raw_documents

# Chunking

In [11]:
BULLET_PREFIXES = ("- ", "* ", "\u2022 ", "\u25cf ")


def _is_heading_line(line: str) -> bool:
    return line.startswith("#") or line.startswith("[Table") or line.startswith("[Speaker notes")


def _is_bullet_line(line: str) -> bool:
    return line.lstrip().startswith(BULLET_PREFIXES)


def _presegment_structure_aware(text: str) -> List[str]:
    """Group lines into structural blocks so headings stay attached to the
    paragraph that follows them, and consecutive bullet-list lines stay
    together as one block, before the recursive splitter ever sees them.
    """
    lines = text.split("\n")
    blocks: List[str] = []
    current: List[str] = []
    current_is_bullets = False

    def flush():
        if current:
            blocks.append("\n".join(current))
            current.clear()

    for line in lines:
        if not line.strip():
            flush()
            current_is_bullets = False
            continue
        if _is_heading_line(line):
            flush()
            current.append(line)
            current_is_bullets = False
            continue
        if _is_bullet_line(line):
            if current and not current_is_bullets and not _is_heading_line(current[-1]):
                flush()
            current.append(line)
            current_is_bullets = True
            continue
        if current_is_bullets:
            flush()
        current.append(line)
        current_is_bullets = False

    flush()
    return [b for b in blocks if b.strip()]


def _split_by_separator(text: str, separator: str) -> List[str]:
    if separator == "":
        return list(text)
    return text.split(separator)


def recursive_character_split(text: str, chunk_size: int = None, chunk_overlap: int = None) -> List[str]:
    """A dependency-free recursive character splitter modeled after
    LangChain's RecursiveCharacterTextSplitter, using a cascading list of
    separators from coarse to fine granularity. Structural blocks
    (headings + following paragraph, bullet-list runs) are pre-merged so
    the splitter is much less likely to sever a heading from its body text
    or break a bullet list mid-list. Identical algorithm to app.py.
    """
    chunk_size = chunk_size or CHUNK_SIZE
    chunk_overlap = chunk_overlap if chunk_overlap is not None else CHUNK_OVERLAP
    separators = ["\n\n", "\n", ". ", " ", ""]

    def _split(text_block: str, seps: List[str]) -> List[str]:
        if len(text_block) <= chunk_size:
            return [text_block] if text_block.strip() else []

        if not seps:
            return [text_block[i:i + chunk_size] for i in range(0, len(text_block), chunk_size)]

        sep = seps[0]
        pieces = _split_by_separator(text_block, sep)

        chunks: List[str] = []
        current = ""
        for piece in pieces:
            candidate = current + (sep if current else "") + piece if sep else current + piece
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current.strip():
                    chunks.append(current)
                if len(piece) > chunk_size:
                    chunks.extend(_split(piece, seps[1:]))
                    current = ""
                else:
                    current = piece
        if current.strip():
            chunks.append(current)
        return chunks

    blocks = _presegment_structure_aware(text)
    merged_blocks: List[str] = []
    current = ""
    for block in blocks:
        candidate = (current + "\n\n" + block) if current else block
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                merged_blocks.append(current)
            current = block
    if current:
        merged_blocks.append(current)

    raw_chunks: List[str] = []
    for block in merged_blocks:
        raw_chunks.extend(_split(block, separators))

    overlapped_chunks = []
    for i, chunk in enumerate(raw_chunks):
        if i == 0 or chunk_overlap <= 0:
            overlapped_chunks.append(chunk)
        else:
            prev_tail = raw_chunks[i - 1][-chunk_overlap:]
            overlapped_chunks.append((prev_tail + chunk)[:chunk_size + chunk_overlap])
    return [c.strip() for c in overlapped_chunks if c.strip()]


def build_chunks_for_file(filename: str, file_type: str, pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Convert loaded page/slide text blocks into metadata-rich chunks,
    identical to app.py's build_chunks_for_file (cleans each page, splits,
    and preserves source_file / page / file_type metadata per chunk).
    """
    chunks = []
    for page_entry in pages:
        cleaned = clean_text(page_entry["text"])
        if not cleaned:
            continue
        text_pieces = recursive_character_split(cleaned)
        for piece in text_pieces:
            chunk_id = str(uuid.uuid4())
            chunks.append({
                "chunk_id": chunk_id,
                "text": piece,
                "source_file": filename,
                "page": page_entry["page"],
                "file_type": file_type,
            })
    print(f"Created {len(chunks)} chunks from '{filename}'.")
    return chunks


def chunk_documents(loaded: Dict[str, Tuple[str, List[Dict[str, Any]]]]) -> List[Dict[str, Any]]:
    """Build chunks for every loaded file and concatenate into one list."""
    all_chunks: List[Dict[str, Any]] = []
    for filename, (file_type, pages) in loaded.items():
        all_chunks.extend(build_chunks_for_file(filename, file_type, pages))
    print(f"Created {len(all_chunks)} chunks from {len(loaded)} document(s).")
    return all_chunks

# Embeddings

In [12]:
# Load the embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Loaded embedding model: {EMBEDDING_MODEL}")


def embed_texts(texts):
    """Generate embeddings for a list of text chunks."""
    if not texts:
        return []

    embeddings = embedding_model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    return embeddings.tolist()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 23761.80it/s]


Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


# Pinecone Vector Database

In [13]:
# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
existing_indexes = [index["name"] for index in pc.list_indexes()]

if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )

    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)

    print(f"Created Pinecone index: {PINECONE_INDEX_NAME}")
else:
    print(f"Using existing Pinecone index: {PINECONE_INDEX_NAME}")

pinecone_index = pc.Index(PINECONE_INDEX_NAME)


def delete_all_vectors() -> None:
    """Remove all vectors from the Pinecone index before indexing a new set of documents."""
    try:
        pinecone_index.delete(delete_all=True)
        print("Removed existing vectors from Pinecone.")
    except Exception as error:
        print(f"Warning: Unable to delete vectors (the index may already be empty): {error}")


def upsert_chunks_to_pinecone(chunks: List[Dict[str, Any]]) -> None:
    """Embed document chunks and store them in Pinecone."""
    if not chunks:
        print("No chunks available for indexing.")
        return

    texts = [chunk["text"] for chunk in chunks]
    embeddings = embed_texts(texts)

    vectors = []
    for chunk, embedding in zip(chunks, embeddings):
        vectors.append({
            "id": chunk["chunk_id"],
            "values": embedding,
            "metadata": {
                "text": chunk["text"][:4000],
                "source_file": chunk["source_file"],
                "page": chunk["page"],
                "file_type": chunk["file_type"],
                "chunk_id": chunk["chunk_id"],
            },
        })

    batch_size = 100
    for start in range(0, len(vectors), batch_size):
        batch = vectors[start:start + batch_size]
        pinecone_index.upsert(vectors=batch)

    print(f"Indexed {len(vectors)} chunks into Pinecone.")


def dense_retrieve(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Retrieve the most relevant document chunks using dense vector search."""
    query_embedding = embed_texts([query])[0]
    result = pinecone_index.query(vector=query_embedding, top_k=top_k, include_metadata=True)

    matches = result.get("matches", []) if isinstance(result, dict) else result.matches

    retrieved_chunks = []
    for match in matches:
        metadata = match["metadata"] if isinstance(match, dict) else match.metadata
        score = match["score"] if isinstance(match, dict) else match.score

        retrieved_chunks.append({
            "chunk_id": metadata.get("chunk_id"),
            "text": metadata.get("text", ""),
            "source_file": metadata.get("source_file"),
            "page": metadata.get("page"),
            "file_type": metadata.get("file_type"),
            "score": float(score),
            "retriever": "dense",
        })

    return retrieved_chunks

Using existing Pinecone index: docsensei-index


# BM25 Retriever

In [14]:
def tokenize(text: str) -> List[str]:
    """Convert text into lowercase alphanumeric tokens."""
    return re.findall(r"[a-z0-9]+", text.lower())


bm25_index: Optional[BM25Okapi] = None
bm25_corpus_tokens: List[List[str]] = []


def build_bm25_index(chunks: List[Dict[str, Any]]) -> Optional[BM25Okapi]:
    """Build or rebuild the BM25 index from the current document chunks."""
    global bm25_index, bm25_corpus_tokens

    if not chunks:
        bm25_index = None
        bm25_corpus_tokens = []
        return None

    bm25_corpus_tokens = [tokenize(chunk["text"]) for chunk in chunks]
    bm25_index = BM25Okapi(bm25_corpus_tokens)

    print(f"Built BM25 index with {len(chunks)} chunks.")

    return bm25_index


def bm25_retrieve(query: str, chunks: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    """Retrieve the most relevant chunks using BM25 keyword search."""
    if bm25_index is None or not chunks:
        return []

    query_tokens = tokenize(query)
    scores = bm25_index.get_scores(query_tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)[:top_k]

    retrieved_chunks = []
    for index in ranked_indices:
        if scores[index] <= 0:
            continue

        chunk = chunks[index]
        retrieved_chunks.append({
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "source_file": chunk["source_file"],
            "page": chunk["page"],
            "file_type": chunk["file_type"],
            "score": float(scores[index]),
            "retriever": "bm25",
        })

    return retrieved_chunks

# Hybrid Retriever

In [15]:
def min_max_normalize(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Normalize retrieval scores to the range [0, 1]."""
    if not results:
        return results

    scores = [result["score"] for result in results]
    minimum_score = min(scores)
    maximum_score = max(scores)
    score_range = maximum_score - minimum_score

    for result in results:
        result["norm_score"] = 1.0 if score_range == 0 else (result["score"] - minimum_score) / score_range

    return results


def hybrid_retrieve(query: str, chunks: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    """Combine dense retrieval and BM25 retrieval using weighted score fusion."""
    dense_results = dense_retrieve(query, top_k=top_k * 2)
    bm25_results = bm25_retrieve(query, chunks, top_k=top_k * 2)

    dense_results = min_max_normalize(dense_results)
    bm25_results = min_max_normalize(bm25_results)

    combined_results: Dict[str, Dict[str, Any]] = {}

    for result in dense_results:
        combined_results[result["chunk_id"]] = dict(result, weighted_score=result["norm_score"] * DENSE_WEIGHT)

    for result in bm25_results:
        if result["chunk_id"] in combined_results:
            combined_results[result["chunk_id"]]["weighted_score"] += result["norm_score"] * BM25_WEIGHT
            combined_results[result["chunk_id"]]["retriever"] = "hybrid"
        else:
            combined_results[result["chunk_id"]] = dict(result, weighted_score=result["norm_score"] * BM25_WEIGHT)

    ranked_chunks = sorted(combined_results.values(), key=lambda chunk: chunk["weighted_score"], reverse=True)

    return ranked_chunks[:top_k]


# Load the CrossEncoder reranker
reranker_model = CrossEncoder(RERANKER_MODEL)

print(f"Loaded reranker model: {RERANKER_MODEL}")


def rerank_chunks(query: str, chunks: List[Dict[str, Any]], top_n: int = None) -> List[Dict[str, Any]]:
    """Improve retrieval quality using CrossEncoder reranking."""
    top_n = top_n or RERANK_TOP_N

    if not chunks:
        return chunks

    try:
        sentence_pairs = [[query, chunk["text"]] for chunk in chunks]
        scores = reranker_model.predict(sentence_pairs)

        for chunk, score in zip(chunks, scores):
            chunk["rerank_score"] = float(score)

        ranked_chunks = sorted(chunks, key=lambda chunk: chunk["rerank_score"], reverse=True)

        return ranked_chunks[:top_n]

    except Exception as error:
        print(f"Warning: Reranking failed: {error}")
        return chunks[:top_n]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 18347.79it/s]


Loaded reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2


# Adaptive RAG Router

In [33]:
def _heuristic_classify_query(query: str) -> str:
    """Fallback query classifier used if the LLM router is unavailable."""

    query = query.strip().lower()
    word_count = len(query.split())

    if query.split() and query.split()[0] in ("it", "that", "this", "they", "them", "those"):
        return "Follow-up"

    if any(word in query for word in ("compare", "difference", "versus", " vs ", "contrast")):
        return "Comparison"

    if any(word in query for word in ("summarize", "summary", "overview", "recap")):
        return "Summarization"

    if any(word in query for word in ("why", "explain", "reasoning", "how come")):
        return "Reasoning"

    if word_count > 20 or "across" in query or "all documents" in query:
        return "Multi-document"

    if word_count <= 3:
        return "Follow-up"

    return "Simple"
def classify_query(query: str) -> str:
    """Classify the query using the LLM router with a heuristic fallback."""
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=0.0,
            messages=[
                {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
                {"role": "user", "content": f"Query: {query}\n\nLabel:"},
            ],
        )

        label = response.choices[0].message.content.strip()

        for known in QUERY_LABELS:
            if known.lower() == label.lower().strip(". "):
                return known

        print("Warning: Using heuristic fallback for query classification.")
        return _heuristic_classify_query(query)

    except Exception:
        print("Warning: LLM router failed. Using heuristic fallback.")
        return _heuristic_classify_query(query)


def adaptive_retrieve(query: str, chunks: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], str]:
    """Retrieve and rerank documents based on the query type."""
    query_type = classify_query(query)
    top_k = ROUTER_TOP_K.get(query_type, 5)

    print(f"Query classified as '{query_type}' (Top-K = {top_k})")

    hybrid_results = hybrid_retrieve(query, chunks, top_k=max(top_k, RERANK_CANDIDATE_POOL))
    reranked_results = rerank_chunks(query, hybrid_results, top_n=min(top_k, RERANK_TOP_N))

    return reranked_results, query_type

# Contextual Compression

In [17]:
def compress_context(query: str, retrieved_chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Compress retrieved chunks by keeping only the content relevant to the query."""
    if not retrieved_chunks:
        return retrieved_chunks

    to_compress = retrieved_chunks[:COMPRESSION_TOP_N]
    passthrough = retrieved_chunks[COMPRESSION_TOP_N:]

    numbered_chunks = "\n\n".join(f"[CHUNK {i}]\n{chunk['text']}" for i, chunk in enumerate(to_compress))

    system_prompt = (
        "You are a context compression engine. For each numbered CHUNK, extract ONLY "
        "the sentences or phrases directly relevant to the user's question. If a chunk "
        "has nothing relevant, output an empty string for it. "
        "Respond ONLY with a JSON object mapping chunk index (as string) to the extracted text, "
        "with no preamble, no markdown fences, and no extra commentary."
    )

    user_prompt = f"Question: {query}\n\n{numbered_chunks}"

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        raw = response.choices[0].message.content.strip()
        raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()

        extracted_map = json.loads(raw)

    except Exception:
        print("Warning: Context compression failed. Using the original retrieved chunks.")
        return retrieved_chunks

    compressed_chunks = []
    for index, chunk in enumerate(to_compress):
        extracted_text = extracted_map.get(str(index), "").strip()

        updated_chunk = dict(chunk)
        updated_chunk["text"] = extracted_text if extracted_text else chunk["text"]

        compressed_chunks.append(updated_chunk)

    return compressed_chunks + passthrough

# Conversation Memory

In [18]:
class ConversationMemory:
    """Store the recent conversation history used for follow-up questions."""

    def __init__(self, window: int = None):
        self.window = window or MEMORY_WINDOW
        self.turns = deque(maxlen=self.window)

    def add(self, user_message: str, assistant_message: str) -> None:
        """Add a new conversation turn."""
        self.turns.append({"user": user_message, "assistant": assistant_message})

    def as_text(self) -> str:
        """Return the conversation history as plain text."""
        if not self.turns:
            return ""

        conversation = []
        for turn in self.turns:
            conversation.append(f"User: {turn['user']}")
            conversation.append(f"Assistant: {turn['assistant']}")

        return "\n".join(conversation)

    def clear(self) -> None:
        """Clear the conversation history."""
        self.turns.clear()

    def __bool__(self) -> bool:
        return bool(self.turns)


memory = ConversationMemory()

# Query Rewriter

In [19]:
STANDALONE_CHECK_PROMPT = (
    "Determine whether the following follow-up question is already a fully "
    "standalone question that can be understood WITHOUT the conversation "
    "history (i.e. it contains no pronouns or references like 'it', 'that', "
    "'those', 'the previous one' that depend on prior context). "
    "Respond with ONLY 'YES' if it is standalone, or 'NO' if it depends on "
    "the conversation history."
)

REWRITE_SYSTEM_PROMPT = (
    "Given the conversation history and a follow-up question, rewrite the follow-up "
    "question to be a fully standalone question that can be understood without the "
    "conversation history. Preserve the original intent. Respond with ONLY the rewritten "
    "question and nothing else."
)


def needs_rewrite(query: str) -> bool:
    """Check whether a follow-up question needs to be rewritten into a standalone query."""
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=0.0,
            messages=[
                {"role": "system", "content": STANDALONE_CHECK_PROMPT},
                {"role": "user", "content": query},
            ],
        )

        verdict = response.choices[0].message.content.strip().upper()

        return not verdict.startswith("YES")

    except Exception as error:
        print(f"Warning: Unable to determine whether the query is standalone ({error}). Rewriting will be used.")
        return True


def rewrite_query(query: str, history_text: str) -> str:
    """Rewrite a follow-up question into a standalone query when required."""
    if not memory:
        return query

    if not needs_rewrite(query):
        print("The query is already standalone.")
        return query

    user_prompt = f"Conversation history:\n{history_text}\n\nFollow-up question: {query}\n\nStandalone question:"

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": REWRITE_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )

        rewritten_query = response.choices[0].message.content.strip()

        if rewritten_query:
            print("Rewrote the follow-up question.")

        return rewritten_query or query

    except Exception as error:
        print(f"Warning: Query rewriting failed ({error}). Using the original query.")
        return query

# Prompt Template

In [20]:
ANSWER_SYSTEM_PROMPT = (
    "You are DocSensei, an enterprise document assistant. You must answer ONLY using the "
    "provided context extracted from the user's uploaded documents. Never use outside or "
    "general knowledge. If the answer cannot be found in the provided context, respond "
    "exactly with: \"I could not find this information in the uploaded documents.\" "
    "Write the answer naturally, in plain prose. Do NOT include citation markers, source "
    "names, or bracketed references inside the answer text itself; citations are handled "
    "separately."
)

# Groq LLM

In [21]:
# Load the Groq client
GROQ_MODEL = "llama-3.3-70b-versatile"
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialised successfully.")

Groq client initialised successfully.


# Complete Adaptive RAG Pipeline

# RAGSession — Orchestrating Ingestion & Answering

The `RAGSession` class ties every earlier building block (document loaders, chunking, embeddings, Pinecone, BM25, the adaptive router, contextual compression, and conversation memory) into two high-level operations:

- **`ingest()`** — loads, chunks, embeds, indexes, and summarizes a new batch of documents.
- **`answer()`** — runs a question through the full Adaptive RAG pipeline and returns an answer with citations.


In [22]:
@dataclass
class RAGSession:
    """
    Holds all state for one document session: the indexed chunks, the
    raw text of each uploaded document, the generated document
    summaries, and the suggested questions produced after ingestion.

    `ingest()` loads and indexes a new batch of files, and `answer()`
    runs a question through the complete Adaptive RAG pipeline.
    """
    chunks: List[Dict[str, Any]] = field(default_factory=list)
    raw_documents: Dict[str, str] = field(default_factory=dict)
    summaries: Dict[str, Any] = field(default_factory=dict)
    suggested_questions: List[Dict[str, str]] = field(default_factory=list)

    def ingest(self, filepaths: List[str]) -> None:
        """
        Ingest a new batch of documents into the session.

        Every upload starts a completely fresh session: previous Pinecone
        vectors, the BM25 index, chunks, and conversation memory are all
        cleared first, so retrieval can never surface a document that was
        already removed. The pipeline runs: load -> chunk -> embed &
        index -> summarize -> generate suggested questions.
        """
        print("Starting new ingestion - clearing previous session state.")
        delete_all_vectors()
        memory.clear()
        self.chunks = []
        self.raw_documents = {}
        self.summaries = {}
        self.suggested_questions = []

        with timeit("Loading"):
            loaded_documents = load_documents(filepaths)

        if not loaded_documents:
            print("Warning: No documents were successfully loaded; ingestion aborted.")
            return

        self.raw_documents = build_raw_documents(loaded_documents)

        with timeit("Chunking"):
            self.chunks = chunk_documents(loaded_documents)

        if not self.chunks:
            print("Warning: No extractable text found in the uploaded files.")
            return

        with timeit("Embedding + Pinecone indexing"):
            upsert_chunks_to_pinecone(self.chunks)

        build_bm25_index(self.chunks)

        with timeit("Summarization"):
            document_summaries: Dict[str, Any] = {}
            for filename, text in self.raw_documents.items():
                try:
                    document_summaries[filename] = summarize_document(filename, text)
                except Exception as error:
                    print(f"Warning: Summary generation failed for '{filename}' ({error}).")
                    document_summaries[filename] = _fallback_summary(text)
            self.summaries = document_summaries

        with timeit("Suggested questions"):
            combined_sample = "\n\n".join(self.raw_documents.values())
            try:
                self.suggested_questions = generate_suggested_questions(combined_sample)
            except Exception as error:
                print(f"Warning: Suggested question generation failed for upload batch ({error}).")
                self.suggested_questions = []

        num_files = len(self.raw_documents)
        print(f"Ingestion complete: {len(self.chunks)} chunks from {num_files} file(s).")

    def answer(self, question: str) -> Dict[str, Any]:
        """
        Answer a question using the full Adaptive RAG pipeline: rewrite
        the query using conversation history, adaptively retrieve
        relevant chunks, compress the context, generate the answer, and
        build the citation list.
        """
        if not self.chunks:
            return {
                "answer": "Please upload one or more documents before asking questions.",
                "citations": [],
            }

        try:
            # 1. Rewrite the query using conversation history (skipped if already standalone)
            history_text = memory.as_text()
            standalone_query = rewrite_query(question, history_text)

            # 2. Adaptive retrieval: LLM router -> hybrid retrieve -> CrossEncoder rerank
            retrieved_chunks, query_type = adaptive_retrieve(standalone_query, self.chunks)

            # 3. Contextual compression (top-N only)
            compressed_chunks = compress_context(standalone_query, retrieved_chunks)

            # 4. Generate the answer
            answer_text = generate_answer(standalone_query, compressed_chunks)

            # 5. Build citations (returned separately, never injected into the answer)
            citation_list = build_citations(compressed_chunks)

            # 6. Update conversation memory
            memory.add(question, answer_text)

            return {
                "answer": answer_text,
                "citations": citation_list,
                "query_type": query_type,
                "rewritten_query": standalone_query,
            }

        except Exception as error:
            print(f"Error: Something went wrong while answering the question ({error}).")
            return {
                "answer": "An error occurred while generating the answer. Please try again.",
                "citations": [],
            }

**Generating the final answer**

Given the compressed context chunks and the (possibly rewritten) query, ask the Groq LLM to produce a grounded, hallucination-resistant answer.

In [23]:
def generate_answer(query: str, context_chunks: List[Dict[str, Any]]) -> str:
    """
    Generate a natural-language answer grounded strictly in the retrieved
    context chunks. The temperature is fixed at GROQ_TEMPERATURE (0.1) to
    keep answers factual and minimize hallucination.
    """
    if not context_chunks:
        return "I could not find this information in the uploaded documents."

    context_text = "\n\n---\n\n".join(c["text"] for c in context_chunks if c["text"].strip())
    if not context_text.strip():
        return "I could not find this information in the uploaded documents."

    user_prompt = f"Context from uploaded documents:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        return response.choices[0].message.content.strip()
    except Exception as error:
        print(f"Error: Answer generation failed ({error}).")
        return "An error occurred while generating the answer. Please try again."


**Building citations**

Citations are tracked separately from the answer text so the UI can display sources (file, page/slide, chunk id) without cluttering the generated prose.

In [24]:
def build_citations(context_chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Build a citation list referencing the source file, page/slide number,
    and chunk id for each context chunk used to generate the answer.
    Citations are kept separate and are never injected into the answer
    text itself.
    """
    citation_list = []
    seen_chunks = set()
    for chunk in context_chunks:
        key = (chunk.get("source_file"), chunk.get("page"), chunk.get("chunk_id"))
        if key in seen_chunks:
            continue
        seen_chunks.add(key)
        snippet = chunk.get("text", "")[:220]
        citation_list.append({
            "source_file": chunk.get("source_file"),
            "page": chunk.get("page"),
            "chunk_id": chunk.get("chunk_id"),
            "file_type": chunk.get("file_type"),
            "snippet": snippet,
        })
    return citation_list

**Creating the session object**

A single `RAGSession` instance holds the state for the notebook run — call `session.ingest(...)` to index documents, then `session.answer(...)` to ask questions.

In [25]:
session = RAGSession()


# Document Summarizer

In [26]:
SUMMARY_SYSTEM_PROMPT = (
    "You are a precise document summarization assistant. Summarize the given document "
    "into a structured JSON object with EXACTLY these keys:\n"
    '  "executive_summary": 2-4 sentence high-level overview.\n'
    '  "key_topics": array of 3-7 short topic strings.\n'
    '  "important_facts": array of 3-6 concise factual statement strings.\n'
    '  "key_numbers": array of notable figures/statistics/dates as strings '
    "(empty array if the document has none).\n"
    '  "conclusion": 1-3 sentence closing takeaway.\n'
    "Respond with ONLY the JSON object, no preamble, no markdown fences."
)


def _fallback_summary(text: str) -> Dict[str, Any]:
    """
    Return a minimal structured summary used only when the LLM call
    fails, so downstream code never has to special-case a missing
    summary.
    """
    return {
        "executive_summary": "Summary unavailable due to an internal error.",
        "key_topics": [],
        "important_facts": [],
        "key_numbers": [],
        "conclusion": "",
    }


def summarize_document(filename: str, full_text: str) -> Dict[str, Any]:
    """
    Generate a structured summary for a single uploaded document,
    covering an executive summary, key topics, important facts, key
    numbers, and a conclusion.
    """
    truncated_text = full_text[:12000]  # guard against oversized prompts
    user_prompt = f"Document: {filename}\n\nContent:\n{truncated_text}\n\nJSON summary:"

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        raw_response = response.choices[0].message.content.strip()
        raw_response = re.sub(r"^```(json)?|```$", "", raw_response, flags=re.MULTILINE).strip()
        parsed_summary = json.loads(raw_response)
        return {
            "executive_summary": parsed_summary.get("executive_summary", ""),
            "key_topics": parsed_summary.get("key_topics", []) or [],
            "important_facts": parsed_summary.get("important_facts", []) or [],
            "key_numbers": parsed_summary.get("key_numbers", []) or [],
            "conclusion": parsed_summary.get("conclusion", ""),
        }
    except Exception as error:
        print(f"Warning: Summarization failed for file '{filename}' ({error}).")
        return _fallback_summary(full_text)

# Suggested Questions

In [27]:
SUGGESTED_QUESTIONS_SYSTEM_PROMPT = (
    "Based on the provided document content, generate exactly 5 insightful questions "
    "that a user could ask and that would be answerable from this content, one for "
    "EACH of the following difficulty/category levels, in this order: "
    "Basic, Intermediate, Advanced, Comparison, Analytical. "
    'Respond ONLY with a JSON array of 5 objects, each shaped as '
    '{"category": "...", "question": "..."}, no preamble, no markdown fences.'
)


def generate_suggested_questions(all_text_sample: str) -> List[Dict[str, str]]:
    """
    Generate exactly 5 suggested questions spanning the Basic,
    Intermediate, Advanced, Comparison, and Analytical categories, based
    on a sample of the ingested document text.
    """
    truncated_text = all_text_sample[:12000]
    user_prompt = f"Content:\n{truncated_text}"

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": SUGGESTED_QUESTIONS_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        raw_response = response.choices[0].message.content.strip()
        raw_response = re.sub(r"^```(json)?|```$", "", raw_response, flags=re.MULTILINE).strip()
        questions = json.loads(raw_response)
        if isinstance(questions, list):
            cleaned_questions = []
            for question in questions[:5]:
                if isinstance(question, dict) and question.get("question"):
                    cleaned_questions.append({
                        "category": question.get("category", ""),
                        "question": question["question"],
                    })
                elif isinstance(question, str):
                    cleaned_questions.append({"category": "", "question": question})
            return cleaned_questions
        return []
    except Exception as error:
        print(f"Warning: Suggested question generation failed ({error}).")
        return []

# Resetting the Session

Use `reset_session()` to wipe all indexed data (Pinecone vectors, BM25 index, chunks, memory, summaries, and suggested questions) without ingesting a new batch of documents.

In [28]:
def reset_session() -> None:
    """
    Completely reset the application state: wipe all Pinecone vectors,
    clear the BM25 index, chunks, conversation memory, and any cached
    summaries or suggested questions. Call this any time you want a
    clean slate without immediately re-ingesting new documents.
    """
    delete_all_vectors()
    memory.clear()
    session.chunks = []
    session.raw_documents = {}
    session.summaries = {}
    session.suggested_questions = []
    build_bm25_index([])
    print("Reset: application state fully cleared.")

# Ingest Documents

Run the cell below to ingest your documents. Update the `documents` list with the file path(s) you want to upload before running.

In [29]:
# Specify the document(s) to index
documents = ["note.docx"]

# Index the documents
session.ingest(documents)

print("Documents indexed successfully.")
print(f"Total text chunks created: {len(session.chunks)}")

# Display document summaries
if session.summaries:
    print("\nDocument Summaries\n" + "-" * 50)
    for document_name, summary in session.summaries.items():
        print(f"\nDocument: {document_name}")
        print(f"Executive Summary: {summary.get('executive_summary', 'Not available')}")

        if summary.get("key_topics"):
            print(f"Key Topics: {', '.join(summary['key_topics'])}")

        if summary.get("important_facts"):
            print("Important Facts:")
            for fact in summary["important_facts"]:
                print(f"• {fact}")

        if summary.get("key_numbers"):
            print(f"Key Numbers: {', '.join(summary['key_numbers'])}")

        if summary.get("conclusion"):
            print(f"Conclusion: {summary['conclusion']}")

# Display suggested questions
if session.suggested_questions:
    print("\nSuggested Questions\n" + "-" * 50)
    for question in session.suggested_questions:
        category = f"[{question['category']}] " if question.get("category") else ""
        print(f"• {category}{question['question']}")

Starting new ingestion - clearing previous session state.
Removed existing vectors from Pinecone.
Loading 'note.docx'...
Loaded 'note.docx' with 192 content block(s).
Successfully loaded 1 out of 1 document(s).
Loading completed in 0.09 seconds.
Prepared 1 document(s) for chunking.
Created 18 chunks from 'note.docx'.
Created 18 chunks from 1 document(s).
Chunking completed in 0.00 seconds.
Indexed 18 chunks into Pinecone.
Embedding + Pinecone indexing completed in 6.74 seconds.
Built BM25 index with 18 chunks.
Summarization completed in 1.07 seconds.
Suggested questions completed in 0.88 seconds.
Ingestion complete: 18 chunks from 1 file(s).
Documents indexed successfully.
Total text chunks created: 18

Document Summaries
--------------------------------------------------

Document: note.docx
Executive Summary: The document covers the essentials of artificial intelligence, including generative AI, large language models, GANs, diffusion models, and prompt engineering. It also discusses 

# Ask Questions

Run the cell below to start an interactive question-answering loop against the ingested documents. Type `exit` to stop.

In [ ]:
while True:
    question = input("\nAsk your question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    result = session.answer(question)

    print("\n" + "=" * 80)
    print("ANSWER:\n")
    print(result["answer"])

    if result.get("citations"):
        print("\nSources:")
        for c in result["citations"]:
            page_part = f", page {c['page']}" if c.get("page") is not None else ""
            print(f"  [{c['source_file']}{page_part}]")